Урок в прозе: https://proproprogs.ru/python_oop/rezhimy-dostupa-public-private-protected-settery-i-gettery

Телеграм-канал: https://t.me/python_selfedu

Основы механизма инкапсуляции

In [1]:
class Point():

    def __init__(self, x, y):
        self.x = x
        self.y = y

In [2]:
# Мы можем обращаться к локальным свйоствам через экземпляр класса и менять их
pt = Point(1, 2)
print(pt.x)
pt.x = 123
print(pt.x)

1
123


Если мы не хотим, чтобы извне был доступ к атрибутам, то его нужно пометить нижним подчеркиванием, тогда он станет protected, его можно использовать внутри класса и во всех дочерних классах. Если поставить два нижних подчеркивания, он станет private. Без подчеркиваний он public

In [3]:
class Point():

    def __init__(self, x, y):
        self._x = x # protected
        self._y = y # protected

pt = Point(1, 2)
print(pt._x, pt._y)

1 2


Все работает! Значит, свойство protected лишь символизирует и сигнализирует программисту, что к атрибуту лучше не обращаться извне, но никак не огранчиивает этого.

In [10]:
class Point():

    def __init__(self, x, y):
        self.__x = x # protected
        self.__y = y # protected

    def set_coord(self, x, y): # сеттер
        if type(x) in (int, float):
            self.__x = x # protected
            self.__y = y # protected
        else:
            raise ValueError('координаты должны быть числами')

    def get_coord(self): # геттер
        return(self.__x, self.__y)


pt = Point(1, 2)
try:
    print(pt.__x, pt.__y)
except:
    print('доступа к атрибутам __x и __y нет')
pt.set_coord(10, 20)
print(pt.__dict__)

print(pt.get_coord())

доступа к атрибутам __x и __y нет


ValueError: координаты должны быть числами

То есть обратиться напрямую мы не смогли, а через отдельынй метод смогли изменить значения атрибутов. А еще через другой метод смогли получить значения атрибутов. Такие методы называются сеттер и геттер (интерфейсные методы). Они нужны дял реализации принципа инкапсуляции. И чтобы случайно не нарушить целостность класса. лучше с ним взаимодействовать через специальные методы. Сеттер может включатьв себя дополнительные проверки

Теперь добавим приватный меод для проверки корректности координат

In [13]:
class Point():

    def __init__(self, x, y):
        self.__x = self.__y = 0
        if self.__check_value(x) and self.__check_value(y):
            self.__x = x # protected
            self.__y = y # protected

    @classmethod
    def __check_value(cls, arg):
        return type(arg) in (int, float)

    def set_coord(self, x, y): # сеттер
        if self.__check_value(x) and self.__check_value(y):
            self.__x = x # protected
            self.__y = y # protected
        else:
            raise ValueError('координаты должны быть числами')

    def get_coord(self): # геттер
        return(self.__x, self.__y)


pt = Point(1, 2)
pt.set_coord(10, 20)
print(pt.get_coord())

(10, 20)


Обратиться напрямую к приватным свойствам мы не можем, посмотрим, какие вообще есть атрибуты в этом экземпляре

In [15]:
print(dir(pt))

['_Point__check_value', '_Point__x', '_Point__y', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'get_coord', 'set_coord']


Приватные атрибуты здесь доступны через имя класса. То есть на самом деле мы можем обратиться к приватным свойствам и методам

In [16]:
pt._Point__x

10

Если есть задача наглухо спрятать какие-то атрибуты, то можно использовать accesify

In [6]:
from accessify import private, protected

class Point():

    def __init__(self, x, y):
        self.__x = self.__y = 0
        if self.check_value(x) and self.check_value(y):
            self.__x = x # protected
            self.__y = y # protected

    @private
    @classmethod
    def check_value(cls, arg):
        return type(arg) in (int, float)

    def set_coord(self, x, y): # сеттер
        if self.check_value(x) and self.check_value(y):
            self.__x = x # protected
            self.__y = y # protected
        else:
            raise ValueError('координаты должны быть числами')

    def get_coord(self): # геттер
        return(self.__x, self.__y)


pt = Point(1, 2)
pt.check_value(5)


InaccessibleDueToItsProtectionLevelException: Point.check_value() is inaccessible due to its protection level

In [7]:
print(dir(pt))

['_Point__x', '_Point__y', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'check_value', 'get_coord', 'set_coord']


Даже если сделать метод check_value публичным, наличие декоратора @private не позволит вообще никак к нему обратиться. Он даже пропадает из __dict__